In [1]:
import os
import glob
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import random
import cv2
import numbers
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Union
from dataclasses import dataclass, asdict
from torchsummary import summary
import math
from einops import rearrange, repeat, einsum
def count_params(model):
    """ Count model trainable parameters """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Number of available cpu: {os.cpu_count()}')


Using device: cuda
Number of available cpu: 4


In [2]:
# Install monarch-attention package
!git clone https://github.com/cjyaras/monarch-attention
!pip install evaluate torchtnt torch-geometric
!pip install ./monarch-attention --no-deps


Cloning into 'monarch-attention'...
remote: Enumerating objects: 1540, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 1540 (delta 81), reused 136 (delta 76), pack-reused 1382 (from 1)
Receiving objects: 100% (1540/1540), 34.46 MiB | 29.38 MiB/s, done.
Resolving deltas: 100% (1023/1023), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not c

In [ ]:
!pip install einops
!pip install torchsummary
!pip install thop


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 3.1 MB/s eta 0:00:00


# Mobile Sketch2Image (MS2I) Network Implementation

## Common Blocks

In [10]:
class UpSample(nn.Module):
    """ UpSampling block using PixelShuffle """
    def __init__(self, filters=64):
        super().__init__()
        self.conv = nn.Conv2d(filters, filters * 2, kernel_size=1, stride=1, padding=0, bias=True)
        self.pixel_shuffle = nn.PixelShuffle(upscale_factor=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.pixel_shuffle(x)
        return x

## DownSampling block
class DownSample(nn.Module):
    """ DownSampling block using PixelUnshuffle """
    def __init__(self, filters=64):
        super().__init__()
        self.conv = nn.Conv2d(filters, filters // 2, kernel_size=1, stride=1, padding=0, bias=True)
        self.pixel_unshuffle = nn.PixelUnshuffle(downscale_factor=2)

    def forward(self, x):
        """ SHAPE (B, C, H, W) -> SHAPE (B, C/4, H/2, W/2) """
        x = self.conv(x)
        x = self.pixel_unshuffle(x)
        return x

class ConvDown(nn.Module):
    """ DownSampling block using strided convolution """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, bias=True)

    def forward(self, x):
        x = self.conv(x)
        return x

class ConvUp(nn.Module):
    """ UpSampling block using Upsample + convolution """
    def __init__(self, in_channels, out_channels, out_shape=None, scale_factor=None):
        super().__init__()
        assert (out_shape is not None) ^ (scale_factor is not None), "Either out_shape or scale_factor must be provided, but not both."
        if out_shape:
            self.out_shape = out_shape
            self.upsample = nn.Upsample(out_shape, mode='bilinear', align_corners=False)
        else:
            self.upsample = nn.Upsample(scale_factor=scale_factor, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        return x

def shape_estimation(h, w, kernel_size=3, stride=1, padding=1):
    """ Estimate the output shape after a convolutional layer. """
    out_h = (h + 2*padding - kernel_size) // stride + 1
    out_w = (w + 2*padding - kernel_size) // stride + 1
    return out_h, out_w

In [11]:
# Basic blocks
class DConvBlock(nn.Module):
    """ Custom Depthwise Convolution Block """
    def __init__(self, inshape, dim=64, expansion_factor=1.0, bias=False):
        super().__init__()
        hidden_features = int(dim*expansion_factor)
        self.conv = nn.Conv2d(inshape, hidden_features, kernel_size=1, bias=bias)
        self.depthwise = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, stride=1, padding=1, groups=hidden_features, bias=bias)

    def forward(self, x):
        x = self.conv(x)
        x = self.depthwise(x)
        return x

# Custom LayerNormalization
class BiasFree_LayerNorm(nn.Module):
    """ Bias-Free Layer Normalization """
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        x = x.contiguous() 
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma+1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    """ With-Bias Layer Normalization """
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        x = x.contiguous() 
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma+1e-5) * self.weight + self.bias
    
class LayerNorm(nn.Module):
    """ Layer Normalization supporting two types: BiasFree and WithBias """
    def __init__(self, dim, LayerNorm_type, out_4d=True):
        super().__init__()
        if LayerNorm_type =='BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)
        self.out_4d = out_4d

    def to_3d(self, x):
        # Convert (B, C, H, W) to (B, H*W, C)
        if len(x.shape) == 3:
            return x
        elif len(x.shape) == 4:
            return rearrange(x, 'b c h w -> b (h w) c')
        else:
            raise ValueError("Input must be a 3D or 4D tensor")
    
    def to_4d(self, x, h, w):
        # Convert (B, H*W, C) to (B, C, H, W)
        if len(x.shape) == 4:
            return x
        elif len(x.shape) == 3:
            return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)
        else:
            raise ValueError("Input must be a 3D or 4D tensor")

    def forward(self, x):
        if self.out_4d:
            h, w = x.shape[-2:]
            return self.to_4d(self.body(self.to_3d(x)), h, w)
        else:
            return self.body(x)

## Reparameterization Block (RepBlock)

In [12]:
class RepConv3(nn.Module):
    def __init__(self, in_channels, out_channels, groups, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy
        self.reparam = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups)
        if not deploy:
            self.conv_3x3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups)
            self.conv_1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=groups)
            self.conv_1x3 = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 3), padding=(0, 1), groups=groups)
            self.conv_3x1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1), padding=(1, 0), groups=groups)
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=groups, bias=False)
            self.conv_3x3_branch = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in ['conv_3x3','conv_1x1','conv_1x3','conv_3x1', 'conv_1x1_branch', 'conv_3x3_branch']:
            if hasattr(self, name):
                delattr(self, name)

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        # Extract weights and biases
        conv_3x3_w, conv_3x3_b = self.conv_3x3.weight, self.conv_3x3.bias
        conv_1x1_w, conv_1x1_b = self.conv_1x1.weight, self.conv_1x1.bias
        conv_1x3_w, conv_1x3_b = self.conv_1x3.weight, self.conv_1x3.bias
        conv_3x1_w, conv_3x1_b = self.conv_3x1.weight, self.conv_3x1.bias
        conv_1x1_branch_w, conv_3x3_branch_w = self.conv_1x1_branch.weight, self.conv_3x3_branch.weight
        # Pad the smaller kernels to 3x3
        conv_1x1_w_pad = F.pad(conv_1x1_w, [1, 1, 1, 1])
        conv_1x3_w_pad = F.pad(conv_1x3_w, [0, 0, 1, 1])
        conv_3x1_w_pad = F.pad(conv_3x1_w, [1, 1, 0, 0])
        if self.groups == 1:
            conv_1x1_3x3_w_pad = F.conv2d(conv_3x3_branch_w, conv_1x1_branch_w.permute(1, 0, 2, 3))
        else:
            w_slices = []
            conv_1x1_branch_w_T = conv_1x1_branch_w.permute(1, 0, 2, 3)
            in_channels_per_group = self.in_channels // self.groups
            out_channels_per_group = self.out_channels // self.groups
            for g in range(self.groups):
                # Slice the transposed 1x1 weights for this group's channels
                conv_1x1_branch_w_T_slice = conv_1x1_branch_w_T[:, g*in_channels_per_group:(g+1)*in_channels_per_group, :, :]
                # Slice the 3x3 weights for this group's output channels
                conv_3x3_branch_w_slice = conv_3x3_branch_w[g*out_channels_per_group:(g+1)*out_channels_per_group, :, :, :]
                w_slices.append(F.conv2d(conv_3x3_branch_w_slice, conv_1x1_branch_w_T_slice))
            conv_1x1_3x3_w_pad = torch.cat(w_slices, dim=0)
        # Fuse weights and biases
        conv_w = conv_3x3_w + conv_1x1_w_pad + conv_1x3_w_pad + conv_3x1_w_pad + conv_1x1_3x3_w_pad
        if conv_3x3_b is None:
            conv_3x3_b = torch.zeros(self.out_channels, device=conv_w.device)
        conv_b = conv_3x3_b + conv_1x1_b + conv_1x3_b + conv_3x1_b
        self.reparam.weight.data.copy_(conv_w)
        self.reparam.bias.data.copy_(conv_b)
        # Delete the original branches
        if delete_branches:
            self._delete_branches()
        # Set deploy flag
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        else:
            return self.conv_3x3(x) + self.conv_1x1(x) + self.conv_1x3(x) + self.conv_3x1(x) + self.conv_3x3_branch(self.conv_1x1_branch(x))

# # Test repconv
# conv = RepConv3(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

In [13]:
class RepConv5(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy
        self.reparam = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)

        if not deploy:
            # Main branches
            self.conv_5x5 = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)
            self.conv_3x3 = nn.Conv2d(in_channels, out_channels, 3, padding=1, groups=groups)
            self.conv_1x1 = nn.Conv2d(in_channels, out_channels, 1, groups=groups)
            # Asymmetric branches
            self.conv_1x5 = nn.Conv2d(in_channels, out_channels, (1,5), padding=(0,2), groups=groups)
            self.conv_5x1 = nn.Conv2d(in_channels, out_channels, (5,1), padding=(2,0), groups=groups)
            self.conv_1x3 = nn.Conv2d(in_channels, out_channels, (1,3), padding=(0,1), groups=groups)
            self.conv_3x1 = nn.Conv2d(in_channels, out_channels, (3,1), padding=(1,0), groups=groups)
            self.conv_3x5 = nn.Conv2d(in_channels, out_channels, (3,5), padding=(1,2), groups=groups)
            self.conv_5x3 = nn.Conv2d(in_channels, out_channels, (5,3), padding=(2,1), groups=groups)
            # Sequential branch
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, 1, groups=groups, bias=False)
            self.conv_5x5_branch = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in [
            'conv_5x5','conv_3x3','conv_1x1',
            'conv_1x5','conv_5x1',
            'conv_1x3','conv_3x1',
            'conv_3x5','conv_5x3',
            'conv_1x1_branch','conv_5x5_branch'
        ]:
            if hasattr(self, name):
                delattr(self, name)

    def _pad_to_5x5(self, w):
        _, _, h, w_k = w.shape
        pad_h = 5 - h
        pad_w = 5 - w_k
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        return F.pad(w, [pad_left, pad_right, pad_top, pad_bottom])

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        def get_wb(conv):
            return conv.weight, conv.bias if conv.bias is not None else torch.zeros(self.out_channels, device=conv.weight.device)
        W = 0
        B = 0
        # Helper to accumulate
        def add_branch(w, b):
            nonlocal W, B
            W = W + w
            B = B + b
        # Main kernels
        w, b = get_wb(self.conv_5x5)
        add_branch(w, b)
        w, b = get_wb(self.conv_3x3)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_1x1)
        add_branch(self._pad_to_5x5(w), b)
        # Asymmetric
        w, b = get_wb(self.conv_1x5)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_5x1)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_1x3)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_3x1)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_3x5)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_5x3)
        add_branch(self._pad_to_5x5(w), b)
        # Sequential 1x1 -> 5x5
        w1 = self.conv_1x1_branch.weight
        w2 = self.conv_5x5_branch.weight
        if self.groups == 1:
            w_seq = F.conv2d(w2, w1.permute(1,0,2,3))
        else:
            w_slices = []
            w1_T = w1.permute(1,0,2,3)
            icpg = self.in_channels // self.groups
            ocpg = self.out_channels // self.groups
            for g in range(self.groups):
                w1_slice = w1_T[:, g*icpg:(g+1)*icpg]
                w2_slice = w2[g*ocpg:(g+1)*ocpg]
                w_slices.append(F.conv2d(w2_slice, w1_slice))
            w_seq = torch.cat(w_slices, dim=0)
        add_branch(w_seq, torch.zeros(self.out_channels, device=w_seq.device))
        self.reparam.weight.data.copy_(W)
        self.reparam.bias.data.copy_(B)
        if delete_branches:
            self._delete_branches()
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        return (
            self.conv_5x5(x)
            + self.conv_3x3(x)
            + self.conv_1x1(x)
            + self.conv_1x5(x)
            + self.conv_5x1(x)
            + self.conv_1x3(x)
            + self.conv_3x1(x)
            + self.conv_3x5(x)
            + self.conv_5x3(x)
            + self.conv_5x5_branch(self.conv_1x1_branch(x))
        )

# # Test RepConv5
# conv = RepConv5(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

In [14]:
class RepConv7(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy

        self.reparam = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups)

        if not deploy:
            # Main
            self.conv_7x7 = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups)
            # Directional
            self.conv_7x1 = nn.Conv2d(in_channels, out_channels, (7,1), padding=(3,0), groups=groups)
            self.conv_1x7 = nn.Conv2d(in_channels, out_channels, (1,7), padding=(0,3), groups=groups)
            # Mixed large
            self.conv_7x5 = nn.Conv2d(in_channels, out_channels, (7,5), padding=(3,2), groups=groups)
            self.conv_5x7 = nn.Conv2d(in_channels, out_channels, (5,7), padding=(2,3), groups=groups)
            # Mid
            self.conv_5x5 = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)
            # Small directional
            self.conv_1x5 = nn.Conv2d(in_channels, out_channels, (1,5), padding=(0,2), groups=groups)
            self.conv_5x1 = nn.Conv2d(in_channels, out_channels, (5,1), padding=(2,0), groups=groups)
            # Sequential branch
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, 1, groups=groups, bias=False)
            self.conv_7x7_branch = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in [
            'conv_7x7',
            'conv_7x1','conv_1x7',
            'conv_7x5','conv_5x7',
            'conv_5x5',
            'conv_1x5','conv_5x1',
            'conv_1x1_branch','conv_7x7_branch'
        ]:
            if hasattr(self, name):
                delattr(self, name)

    def _pad_to_7x7(self, w):
        _, _, h, w_k = w.shape
        pad_h = 7 - h
        pad_w = 7 - w_k
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        return F.pad(w, [pad_left, pad_right, pad_top, pad_bottom])

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        def get_wb(conv):
            return conv.weight, conv.bias if conv.bias is not None else torch.zeros(self.out_channels, device=conv.weight.device)
        W = torch.zeros_like(self.reparam.weight)
        B = torch.zeros_like(self.reparam.bias)
        def add_branch(w, b):
            nonlocal W, B
            W += w
            B += b
        # Main
        w, b = get_wb(self.conv_7x7)
        add_branch(w, b)
        # Directional
        for conv in [self.conv_7x1, self.conv_1x7]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Mixed large
        for conv in [self.conv_7x5, self.conv_5x7]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Mid
        w, b = get_wb(self.conv_5x5)
        add_branch(self._pad_to_7x7(w), b)
        # Small directional
        for conv in [self.conv_1x5, self.conv_5x1]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Sequential 1x1 → 7x7
        w1 = self.conv_1x1_branch.weight
        w2 = self.conv_7x7_branch.weight
        if self.groups == 1:
            w_seq = F.conv2d(w2, w1.permute(1,0,2,3))
        else:
            w_slices = []
            w1_T = w1.permute(1,0,2,3)
            icpg = self.in_channels // self.groups
            ocpg = self.out_channels // self.groups
            for g in range(self.groups):
                w1_slice = w1_T[:, g*icpg:(g+1)*icpg]
                w2_slice = w2[g*ocpg:(g+1)*ocpg]
                w_slices.append(F.conv2d(w2_slice, w1_slice))
            w_seq = torch.cat(w_slices, dim=0)
        add_branch(w_seq, torch.zeros(self.out_channels, device=w_seq.device))
        self.reparam.weight.data.copy_(W)
        self.reparam.bias.data.copy_(B)
        if delete_branches:
            self._delete_branches()
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        return (
            self.conv_7x7(x)
            + self.conv_7x1(x)
            + self.conv_1x7(x)
            + self.conv_7x5(x)
            + self.conv_5x7(x)
            + self.conv_5x5(x)
            + self.conv_1x5(x)
            + self.conv_5x1(x)
            + self.conv_7x7_branch(self.conv_1x1_branch(x))
        )

# # Test RepConv7
# conv = RepConv7(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

## Attention Block (MonarchAttn)

In [15]:
# Thêm đường dẫn tới folder monarch-attention vào hệ thống
sys.path.append(os.path.abspath('monarch-attention'))

from ma.monarch_attention import MonarchAttention
@dataclass
class RepAttnConfig:
    dim: int
    num_heads: int = 8
    block_size: int = 16
    num_steps: int = 2
    pad_type: str = "pre"
    impl: str = "torch"
    deploy: bool = False

class RepAttn(nn.Module):
    """ Re-parameterizable Attention Block using MonarchAttention as the core attention mechanism."""
    def __init__(self, dim, num_heads=8, block_size=14, num_steps=1, pad_type="pre", impl="torch", deploy=False):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1)
        self.monarch_attn = MonarchAttention(
            block_size=block_size,
            num_steps=num_steps,
            pad_type=pad_type,
            impl=impl
        )
        if deploy:
            self.attn_fn = self.monarch_attn
        else:
            self.attn_fn = self.common_attn
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)
        self.deploy = deploy

    def common_attn(self, q, k, v):
        """ Scaled Dot-Product Attention """
        scale = (q.shape[-1]) ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        out = attn @ v
        return out

    @torch.no_grad()
    def fuse(self):
        if not self.deploy:
            self.attn_fn = self.monarch_attn
            self.deploy = True

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=1)
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        attn_out = self.attn_fn(q, k, v)
        attn_out = rearrange(attn_out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=H, w=W)
        out = self.proj(attn_out)
        return out

# # Test RepAttn
# attn = RepAttn(dim=256, num_heads=8, block_size=14, num_steps=1, pad_type="pre", impl="torch", deploy=False).cuda()
# x = torch.randn(1, 256, 45, 31).cuda()
# out = attn(x)
# print(out.shape)

## Simple ConvFFN

In [16]:
@dataclass
class FFNConfig:
    dim: int
    expansion_factor: int = 1
    deploy: bool = False

In [17]:
class RepFFN(nn.Module):
    def __init__(self, dim, expansion_factor=1, deploy=False):
        super().__init__()
        hidden_features = int(dim * expansion_factor)
        self.project_in = RepConv3(dim, hidden_features, groups=1, deploy=deploy)
        self.dwconv = RepConv3(hidden_features, hidden_features*2, groups=hidden_features, deploy=deploy)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1)

    @torch.no_grad()
    def fuse(self):
        self.project_in.fuse()
        self.dwconv.fuse()  


    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

## Model Blocks

In [18]:
class SkipConnection(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim*2, dim, kernel_size=1)

    def forward(self, x1, x2):
        x = torch.cat([x1, x2], dim=1)
        x = self.conv(x)
        return x
    
class RepTransformerBlock(nn.Module):
    def __init__(self, rep_attn_cfg: RepAttnConfig, ffn_cfg: FFNConfig, norm_type='WithBias'):
        super().__init__()
        self.rep_attn = RepAttn(**asdict(rep_attn_cfg))
        self.rep_ffn = RepFFN(**asdict(ffn_cfg))
        self.norm1 = LayerNorm(rep_attn_cfg.dim, norm_type)
        self.norm2 = LayerNorm(rep_attn_cfg.dim, norm_type)

    @torch.no_grad()
    def fuse(self):
        self.rep_attn.fuse()
        self.rep_ffn.fuse()

    def forward(self, x):
        x = x + self.rep_attn(self.norm1(x))
        x = x + self.rep_ffn(self.norm2(x))
        return x
    
class Block(nn.Module):
    def __init__(self, num_block, rep_attn_cfg: RepAttnConfig, ffn_cfg: FFNConfig, norm_type='WithBias'):
        super().__init__()
        self.num_block = num_block
        self.blocks = nn.ModuleList([
            RepTransformerBlock(rep_attn_cfg, ffn_cfg, norm_type) for _ in range(num_block)
        ])

    @torch.no_grad()
    def fuse(self):
        for block in self.blocks:
            block.fuse()

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

In [19]:
class Head(nn.Module):
    def __init__(self, in_channels, out_channels=3, deploy=False, last_act=None):
        super().__init__()
        hidden = max(in_channels // 2, 8)
        self.pre = RepConv3(in_channels, hidden, 1, deploy=deploy)
        self.to_rgb = nn.Conv2d(hidden, out_channels, kernel_size=1)
        self.last_act = last_act if last_act is not None else nn.Identity()

    @torch.no_grad()
    def fuse(self):
        if isinstance(self.pre, RepConv3):
            self.pre.fuse()

    def forward(self, x):
        x = F.gelu(self.pre(x))
        return self.last_act(self.to_rgb(x))

class MS2I(nn.Module):
    """Structure-based disentangled MS2I generator."""

    def __init__(
        self,
        input_shape=(3, 256, 256),
        deploy=False,
        dims=[48, 96, 192, 384],
        num_blocks=[4, 6, 6, 8],
        num_heads=[1, 2, 2, 4],
        bias=True,
        last_act=None,
    ):
        super().__init__()
        assert len(dims) == len(num_blocks) == len(num_heads), "Length of dims, num_blocks and num_heads must be the same"
        self.input_shape = input_shape
        self.deploy = deploy
        self.dims = dims
        self.num_blocks = num_blocks
        self.bias = bias
        self.num_heads = num_heads

        # Structure encoder: sketch is the only input here, so early features stay layout-dominant.
        self.stem = nn.Conv2d(3, dims[0], kernel_size=7, stride=4, padding=3, bias=bias)

        layers = []
        down_convs = []
        for idx in range(len(dims)):
            attn_cfg, ffn_cfg = self.build_cfg(dims[idx], num_heads[idx])
            block = Block(num_blocks[idx], attn_cfg, ffn_cfg, norm_type='WithBias')
            if idx < len(dims) - 1:
                down_convs.append(DownSample(dims[idx]))
            layers.append(block)
        self.bottleneck = layers[-1]
        self.encoder = nn.ModuleList(layers[:-1])
        self.downsample = nn.ModuleList(down_convs)

        layers = []
        up_convs = []
        skip_connections = []
        for stage, idx in enumerate(range(len(dims) - 2, -1, -1)):
            attn_cfg, ffn_cfg = self.build_cfg(dims[idx], num_heads[idx])
            up_convs.append(UpSample(dims[idx + 1]))
            skip_connections.append(SkipConnection(dims[idx]))
            layers.append(Block(num_blocks[idx], attn_cfg, ffn_cfg, norm_type='WithBias'))
        self.decoder = nn.ModuleList(layers)
        self.up_sample = nn.ModuleList(up_convs)
        self.skip = nn.ModuleList(skip_connections)

        self.head = Head(dims[0], out_channels=3, deploy=deploy, last_act=last_act)

    @torch.no_grad()
    def fuse(self):
        for block in self.encoder:
            block.fuse()
        self.bottleneck.fuse()
        for block in self.decoder:
            block.fuse()
        self.head.fuse()

    def build_cfg(self, dim, head):
        attn_cfg = RepAttnConfig(
            dim=dim,
            num_heads=head,
            block_size=16,
            num_steps=2,
            pad_type="pre",
            impl="torch",
            deploy=self.deploy,
        )
        ffn_cfg = FFNConfig(dim=dim, expansion_factor=1)
        return attn_cfg, ffn_cfg

    def forward(self, sketch, return_latents=False):

        x = self.stem(sketch)
        structure_feats = []
        for blk, down in zip(self.encoder, self.downsample):
            x = blk(x)
            structure_feats.append(x)
            x = down(x)

        x = self.bottleneck(x)

        for blk, up, skip in zip(self.decoder, self.up_sample, self.skip):
            x = up(x)
            cur_feat = structure_feats.pop()
            x = skip(x, cur_feat)
            x = blk(x)

        x = F.interpolate(x, size=self.input_shape[1:], mode='bilinear', align_corners=False)
        out = self.head(x)
        if return_latents:
            return out, {'structure': x}
        return out

In [20]:
model_cfg = {
    'input_shape': (3, 256, 256),
    'dims': [32, 64, 128, 256],
    'num_blocks': [1, 2, 2, 4],
    'num_heads': [1, 2, 4, 8],
    'bias': True,
    'last_act': None,
    'deploy': False,
}


Output shape: torch.Size([1, 3, 256, 256])
Structure feature shape: torch.Size([1, 32, 256, 256])


In [23]:
def denormalize_tensor(x):
    return (x.detach().cpu().clamp(-1, 1) + 1.0) / 2.0
import matplotlib.pyplot as plt
import numpy as np
def save_sample_grid(sketch_batch, fake_batch, real_batch, out_path=None, max_items=4):
    n = min(max_items, sketch_batch.size(0), fake_batch.size(0), real_batch.size(0))
    if n == 0:
        return
    sketch_batch = denormalize_tensor(sketch_batch[:n])
    fake_batch = denormalize_tensor(fake_batch[:n])
    real_batch = denormalize_tensor(real_batch[:n])
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)
    column_titles = ['Sketch', 'Generated', 'Real']
    for col, title in enumerate(column_titles):
        axes[0, col].set_title(title)
    for row_idx in range(n):
        for col_idx, tensor in enumerate([sketch_batch, fake_batch, real_batch]):
            img = tensor[row_idx].permute(1, 2, 0).numpy()
            axes[row_idx, col_idx].imshow(img)
            axes[row_idx, col_idx].axis('off')
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


In [24]:
import torch
from pathlib import Path
def load_generator_weights(ckpt_path, generator, strict=True):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
    print(f"Loading weights from {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if 'generator_state_dict' in ckpt:
        generator.load_state_dict(ckpt['generator_state_dict'], strict=strict)
    else:
        # Fallback if checkpoint only contains state dict directly
        generator.load_state_dict(ckpt, strict=strict)
        
    print("Weights loaded successfully.")


## Inference Configuration


In [ ]:
import os
import glob
from pathlib import Path

# Cấu hình đường dẫn
# input_path có thể là đường dẫn tới 1 ảnh (vd: 'sketch.jpg') hoặc 1 folder chứa ảnh (vd: 'sketches/')
input_path = '/kaggle/input/your-sketch-dataset/sketches' 
output_dir = '/kaggle/working/generated_images'
weight_path = '/kaggle/input/your-model-weights/ms2i_weights.pth'

os.makedirs(output_dir, exist_ok=True)


## Load Model


In [ ]:
# Khởi tạo model với deploy=True
model_cfg['deploy'] = True
model = MS2I(**model_cfg).to(DEVICE)

# Load weights
if os.path.exists(weight_path):
    model = load_generator_weights(weight_path, model, DEVICE)
    print("Model weights loaded successfully.")
else:
    print(f"Warning: Weight file not found at {weight_path}")

model.eval()
print("Model is ready for inference.")


## Inference Logic


In [ ]:
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

def process_image(img_path):
    # Load and preprocess
    img = Image.open(img_path).convert('RGB')
    
    # Pad and resize (using the same logic if possible, or simple resize)
    w, h = img.size
    max_side = max(w, h)
    canvas_side = max(max_side, 256)
    pad_left = (canvas_side - w) // 2
    pad_top = (canvas_side - h) // 2
    pad_right = canvas_side - w - pad_left
    pad_bottom = canvas_side - h - pad_top
    
    from PIL import ImageOps
    img_padded = ImageOps.expand(img, border=(pad_left, pad_top, pad_right, pad_bottom), fill=(255, 255, 255))
    img_resized = img_padded.resize((256, 256), Image.BICUBIC)
    
    # To Tensor
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    tensor = transform(img_resized).unsqueeze(0).to(DEVICE)
    return tensor, img_resized

@torch.no_grad()
def infer(input_p, output_d):
    path = Path(input_p)
    if path.is_file():
        image_paths = [path]
    elif path.is_dir():
        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
            image_paths.extend(path.glob(ext))
    else:
        print("Invalid input path.")
        return []

    generated_files = []
    for img_path in tqdm(image_paths, desc="Generating images"):
        tensor, _ = process_image(img_path)
        out_tensor = model(tensor)
        out_tensor = denormalize_tensor(out_tensor)
        
        # Save image
        out_img = out_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
        out_img = (out_img * 255).astype('uint8')
        out_img = Image.fromarray(out_img)
        
        save_path = os.path.join(output_d, f"gen_{img_path.name}")
        out_img.save(save_path)
        generated_files.append((img_path, save_path))
        
    return generated_files

results = infer(input_path, output_dir)
print(f"Generated {len(results)} images.")


## Zip Output


In [ ]:
import shutil

zip_path = '/kaggle/working/generated_results'
shutil.make_archive(zip_path, 'zip', output_dir)
print(f"Saved zipped outputs to {zip_path}.zip")


## Display Results


In [ ]:
import matplotlib.pyplot as plt

num_display = min(5, len(results))
if num_display > 0:
    fig, axes = plt.subplots(num_display, 2, figsize=(10, 5 * num_display))
    if num_display == 1:
        axes = [axes]
        
    for i in range(num_display):
        orig_path, gen_path = results[i]
        orig_img = Image.open(orig_path)
        gen_img = Image.open(gen_path)
        
        axes[i][0].imshow(orig_img)
        axes[i][0].set_title("Input Sketch")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(gen_img)
        axes[i][1].set_title("Generated Image")
        axes[i][1].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No images to display.")
